In [ ]:
import pandas as pd


df = pd.read_csv('Resume.csv')

print("Dataset Dimensions:", df.shape)
print("\nMissing Values per Column:\n", df.isnull().sum())
print("\nFirst 2 Rows of Data:")
print(df.head(2))

In [ ]:

category_counts = df['Category'].value_counts()
print("Distribution of Resume Categories:")
print(category_counts)


df = df.drop(columns=['Resume_html'])
print("\nColumns remaining after drop:", df.columns.tolist())

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

def clean_resume_text(text):
    """Cleans raw text by removing noise, links, emails, and special characters."""
    if not isinstance(text, str):
        return ""
    
    text = text.lower()  
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  
    text = re.sub(r'\S+@\S+', '', text) 
    text = re.sub(r'[^a-zA-Z\s]', '', text) 
    text = re.sub(r'\s+', ' ', text).strip() 
    return text


print("Preprocessing raw text strings...")
df['Clean_Resume'] = df['Resume_str'].apply(clean_resume_text)

tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
print("NLP Preprocessing complete.")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

t
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['Clean_Resume'], 
    df['Category'], 
    test_size=0.20, 
    random_state=42, 
    stratify=df['Category']
)


X_train_vec = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_vec = tfidf_vectorizer.transform(X_test_raw)


classifier_model = LogisticRegression(max_iter=1000)
print("Training the classification model...")
classifier_model.fit(X_train_vec, y_train)


y_pred = classifier_model.predict(X_test_vec)
print(f"\nModel Evaluation Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import pickle

with open('resume_vectorizer.pkl', 'wb') as file:
    pickle.dump(tfidf_vectorizer, file)


with open('resume_classifier.pkl', 'wb') as file:
    pickle.dump(classifier_model, file)



In [ ]:
import pickle
import re


def clean_resume_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) 
    text = re.sub(r'\S+@\S+', '', text)                                    
    text = re.sub(r'[^a-zA-Z\s]', '', text)                                
    return re.sub(r'\s+', ' ', text).strip()                               


try:
    with open('resume_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    with open('resume_classifier.pkl', 'rb') as f:
        model = pickle.load(f)
    print("✓ Model and Vectorizer loaded successfully!\n")
except FileNotFoundError:
    print("❌ Error: Could not find the .pkl files. Make sure to run your training script first.")
    exit()


sample_resume_it = """
Senior Full Stack Engineer with 5 years of experience building web applications. 
Proficient in Python, JavaScript, React, SQL databases, and AWS cloud architecture. 
Managed code repositories using Git and deployed Docker containers into production.
"""


sample_resume_finance = """
Certified Public Accountant (CPA) managing financial ledgers, corporate tax preparation, 
and quarterly audit tracking. Expert in Excel spreadsheets, balance sheets, and budget forecasting.
"""


def test_input(raw_text):
    cleaned_text = clean_resume_text(raw_text)
    vectorized_text = vectorizer.transform([cleaned_text])
    prediction = model.predict(vectorized_text)[0]
    
 
    probabilities = model.predict_proba(vectorized_text)
    max_confidence = max(probabilities[0]) * 100
    
    print(f"--- Testing Resume Text Snippet ---")
    print(f"Predicted Class : {prediction}")
    print(f"Confidence Score: {max_confidence:.2f}%")
    print("-" * 35)

test_input(sample_resume_it)
test_input(sample_resume_finance)